# TF -> mRNA coupling strength inference for *S. cerevisiae*

**Project:** transcription module for a whole-cell model of budding yeast
**Repo:** [github.com/djsegal/tf-coupling-fit](https://github.com/djsegal/tf-coupling-fit)
**Author:** Daniel Segal (with implementation help)
**Status:** v1 prototype, suitable as input to the larger whole-cell model

---

## What this notebook does

Given a directed TF -> substrate regulatory network from Teufel et al. 2019
and the WT-unstressed RNA-seq time course from the same paper, this notebook
estimates per-edge coupling coefficients $\alpha_{ij}$ in the linear model:

$$S_i(t) = \sum_{j \in R(i)} \alpha_{ij} \cdot TF_j(t - \tau)$$

where $S_i(t)$ is mRNA level of substrate gene $i$ at time $t$,
$TF_j$ is mRNA level of regulating transcription factor $j$, and
$\tau \approx 20$ min is the assumed TF-transcript-to-active-protein delay.

The output is a handoff folder (`tf_handoff/`) containing $\alpha$ values
for ~2,400 substrate genes and ~150 TFs, ready to plug into the
transcription module of a larger whole-cell model.

## Who this is for

Three audiences, three reading depths:

1. **You already know the project (Edda, senior collaborators).**
   Skip to section 1. The headline findings and design decisions are in the
   short README block below; everything else is the analysis you'd expect.

2. **You're a student on the whole-cell rebuild team.** Read the
   **Background** section first (just below). It frames why we're doing
   this and where it plugs into the larger model. The notebook itself
   serves as a mini-tutorial on the data, the math, and the Julia idioms.

3. **You're new to Julia.** The **Julia primer** below the Background
   section covers the minimum syntax you need to read the rest. You don't
   need to write Julia to follow along.

## Table of contents

- [Background: what we're doing and why](#background)
- [Julia primer (skip if fluent)](#julia-primer)
- [1. Setup](#1-setup)
- [2. Parameters](#2-parameters)
- [3. Helpers](#3-helpers)
- [4. Data loading](#4-data-loading)
- [5. The optimization](#5-the-optimization)
- [6. Run it](#6-run-it)
- [7. Validation: cell-cycle canaries](#7-validation-cell-cycle-canaries)
- [8. Why NaN handling matters](#8-why-nan-handling-matters)
- [9. Regularization sweep, the L-curve](#9-regularization-sweep-the-l-curve)
- [10. Global fit quality](#10-global-fit-quality)
- [11. Inspect a single substrate](#11-inspect-a-single-substrate)
- [12. Design decisions](#12-design-decisions-with-justifications)
- [13. Handoff for the whole-cell model](#13-handoff-for-the-whole-cell-model)
- [14. What's not done yet](#14-whats-not-done-yet-v2-)
- [Glossary](#glossary)
- [Appendix: re-deriving the CSV files](#appendix-re-deriving-the-csv-files)

## Data sources

The notebook expects two CSV files in `data/` (alongside the notebook in the
repo) or in `~/Downloads/`. Both ship with this repo so it's self-contained:

- **`WT_unstressed_readspermillionreads.csv`** -- Wild-type unstressed
  RNA-seq time course from Teufel et al. 2019, *Scientific Reports* 9:3343,
  [doi:10.1038/s41598-019-39850-7](https://doi.org/10.1038/s41598-019-39850-7).
  6,701 genes by 22 timepoints (0 to 150 min) plus an unsynchronized
  baseline. Values are reads per million reads (RPM).
- **`TF_von_Teufel.csv`** -- Supplementary table 4 from the same paper:
  8,685 directed `tf, relation, substrate` edges over 150 TFs and 3,455
  substrates.

See the Appendix at the bottom for conversion notes if you ever need to
re-derive the CSVs from the original `.ods` and `.numbers` files.

## Headline findings

These are what running this notebook will show you. The reference figures
in `figures_reference/` were produced from a verified Python implementation
of the same logic; the Julia code below should produce numerically
identical results.

1. **The fit recovers textbook cell-cycle biology.** CLN2 -> SWI4 (SBF
   complex), SIC1 -> ACE2, ACE2 -> FKH2, CLB5 -> SWI6/SWI4 (MBF). See the
   canary panel in section 7.
2. **Strict NaN filtering silently breaks the biology.** Of the 150 TFs in
   the network, 57 get dropped if we require zero NaN timepoints,
   including SWI4 itself. With `max_nans = 4` and linear interpolation of
   short gaps, only 30 drop. Under strict filtering, CLN2 looks regulated
   by STE12 (mating pathway) because SWI4 is unavailable. See section 8.
3. **The L1 penalty has a clear elbow** around $w_2 = 5$. Below $w_2 \approx 1$
   the LP finds vertex solutions with $|\alpha|_{\max} \sim 1000$. Above
   $w_2 \approx 50$ residuals climb sharply. We default to $w_2 = 5$. See
   section 9.
4. **Strong oscillators are under-predicted in amplitude.** CLN2 peaks at
   ~25 molecules per cell, but the linear model with the available
   regulators only reaches ~8. This is structural and likely needs either
   a missing regulator (MBP1 is not in the network for CLN2) or a
   non-linear formulation. See sections 7 and 10.

## What's not done yet (v2+)

Cross-validation per substrate. Bootstrap confidence intervals on $\alpha$.
Random-network permutation baseline. A degradation term. Volume
normalization across the cell cycle. Comparison against an intercept-augmented
model (see section 10: the short answer is that an intercept absorbs everything
at $w_2 = 5$ and zeroes the biology, so doc-literal is right for now).
See section 14 for the full list with justifications.

## Caveat on rigor

This is a defensible v1 prototype, not a publication. The two strongest
sanity checks are (a) the textbook cell-cycle TFs come out where they
should, and (b) replacing the network with a random one should and does
fit much worse (informal check, formalised in v2).

<a id="background"></a>
## Background: what we're doing and why

*Optional reading. Skip to section 1 if you already know the project.*

### Why transcription matters for the whole-cell model

A whole-cell model simulates everything a cell does: metabolism, growth,
division, response to environment. To do that, it has to know how much of
each protein is around at each moment. Proteins come from mRNA, mRNA comes
from transcription, and transcription rates depend on which transcription
factors (TFs) are active.

The larger whole-cell project historically had no real transcription
layer: protein synthesis was modeled in bulk, as if every gene got
translated at the same rate. That's fine for steady-state metabolism but
falls apart for any process that depends on which genes are turned on
*when*. The cell cycle is the obvious example: CLN2 needs to peak in G1,
CLB5 in S phase, CLB2 in M phase, and a flat synthesis model can't
capture that.

This notebook fills the gap. For each gene with measured RNA-seq data and
at least one known regulator, we fit how strongly each TF couples to that
gene's transcription rate. The output (an $\alpha$ matrix) plugs directly
into the transcription module of the whole-cell rebuild.

### The Teufel 2019 dataset

[Teufel et al. 2019](https://doi.org/10.1038/s41598-019-39850-7) did exactly
the experiment we need: synchronized WT yeast in the cell cycle and took
RNA-seq snapshots at 22 timepoints from 0 to 150 minutes after alpha-factor
release. They also published a curated TF -> substrate network with 8,685
edges spanning 150 TFs and 3,455 substrate genes. We use both.

### The model in one equation

For each substrate gene $i$ at each timepoint $t$:

$$S_i(t) = \sum_{j \in R(i)} \alpha_{ij} \cdot TF_j(t - \tau)$$

$S_i(t)$ is mRNA level of substrate $i$. $R(i)$ is the set of TFs that
regulate $i$ per the Teufel network. $TF_j$ is mRNA level of TF $j$
(used as a proxy for active TF protein, which we don't have data for).
$\tau = 20$ min is the assumed lag from TF mRNA to active TF protein
ready to drive transcription. $\alpha_{ij}$ is what we're solving for:
the coupling strength.

Linear and additive is the simplest model that can possibly work. It's
also what the project's strategy document specifies. Non-linear (e.g.
multiplicative combinations of TFs) is on the v2+ list; section 7
discusses why the linear formulation under-predicts peak amplitudes and
when that matters.

### How we fit it

For each substrate independently, we set up a linear program with HiGHS
that minimizes the L1 residual plus an L1 penalty on $\alpha$ to encourage
sparsity. L1 + L1 stays a linear program (fast, exact, convex) and the L1
penalty drives small coefficients exactly to zero, which is what we want
for a sparse regulatory network. Section 5 has the details.

### Where this fits in the bigger picture

```
Teufel 2019 RNA-seq + TF network          (input, ships with this repo)
              |
              v
  THIS NOTEBOOK: per-substrate L1 LP fits  (~5 sec on a laptop)
              |
              v
  tf_handoff/                              (output, see section 13)
   - alpha_matrix.csv                       <-- the substrate x TF coupling matrix
   - tf_network_fitted.csv                  <-- the network the fit kept
   - metadata.json + README.md              <-- provenance and conventions
              |
              v
  WHOLE-CELL MODEL transcription module    (built by the class)
```

The transcription module in the larger model loads `tf_handoff/`, uses the
$\alpha$ matrix to compute per-gene transcription rates as a function of
TF levels, and feeds those rates into whatever ODE / Gillespie / FBA
machinery the rest of the model is using.

<a id="julia-primer"></a>
## Julia primer (skip if fluent)

*Optional reading. The minimum you need to read this notebook.*

If you've used Python or MATLAB, Julia will feel mostly familiar. The few
things worth knowing up front:

| Concept | Looks like | Notes |
|---|---|---|
| Variable | `x = 42` | No `var` keyword, no type declaration needed |
| Constant | `const W = 5` | Top-level only. Just signals intent. |
| Function | `function f(x); x + 1; end` or `f(x) = x + 1` | Return value is the last expression |
| Anonymous function | `x -> x + 1` | Like Python's `lambda x: x + 1` |
| Vector | `[1, 2, 3]` :: `Vector{Int}` | 1-indexed (yes, really) |
| Matrix | `[1 2; 3 4]` :: `Matrix{Int}` | Space-separated columns, semicolons for rows |
| Indexing | `A[1, 2]`, `A[1, :]`, `A[1:3, end]` | 1-based; `end` is the last index |
| Range | `1:5` is `[1,2,3,4,5]`, `1:2:10` is `[1,3,5,7,9]` | Half-open ranges via `1:5` (inclusive on both ends, unlike Python) |
| Broadcast | `x .+ 1`, `sqrt.(x)`, `f.(x)` | The dot makes anything element-wise |
| Comprehension | `[x^2 for x in 1:5]` | Like Python |
| Dict | `Dict("a" => 1, "b" => 2)` | The `=>` is a `Pair` |
| String interpolation | `"$x is $(x+1)"` | `$expr` or `$(complex_expr)` inside `"..."` |
| Symbol | `:foo` | Like an interned string. Used a lot in keyword args. |
| `using X` vs `import X` | `using X` brings X's exported names into scope; `import X` requires `X.name` | We use `using` everywhere |

Two Julia-specific idioms in this notebook worth flagging:

- **Multi-line nested loops with commas**: `for x in A, y in B; body; end` is
  the same as a nested `for x in A; for y in B; body; end; end`. We use this
  pattern in section 13 for compactness.
- **The `do` block**: `open(path, "w") do fh; write(fh, "..."); end`. The
  function `open` takes a callback. Whatever you put after `do args` is the
  body. Equivalent to Python's `with open(...) as fh:`.

If something else looks weird, ask. Julia has a pretty good
[official manual](https://docs.julialang.org/) and the
[Cheat Sheet](https://cheatsheet.juliadocs.org/) covers most syntax in
two pages.

## 1. Setup

If running this notebook for the first time, uncomment the `Pkg.add`
block in the next cell to install dependencies. After that, the
`using` block alone is sufficient each session.

In [ ]:
# One-time setup. Uncomment to install dependencies.
# using Pkg
# Pkg.add(["DataFrames", "CSV", "JuMP", "HiGHS",
#          "Statistics", "LinearAlgebra", "Plots", "Printf",
#          "ZipFile", "XLSX"])   # last two only used by section 13

using DataFrames
using CSV
using JuMP, HiGHS
using Statistics, LinearAlgebra
using Plots
using Printf


## 2. Parameters

All knobs in one place. The defaults reflect the analysis decisions
discussed in the README at the top of this notebook.

In [ ]:
const MOLECULES_PER_CELL = 60_000
const RPM_TO_MOLECULES   = MOLECULES_PER_CELL / 1_000_000   # = 0.06

const DEFAULT_TAU      = 20.0   # TF -> protein delay (min)
const DEFAULT_W2       = 5.0    # L1 weight on alpha (elbow of L-curve)
const DEFAULT_MAX_NANS = 4      # NaN tolerance per gene (of 22 timepoints)
nothing  # suppress display of last value

## 3. Helpers

### Linear interpolation with edge clamping

We use this in two places: (a) filling short NaN gaps in time courses,
and (b) sampling TF time courses at $t - \tau$. The behavior matches
`numpy.interp`: queries outside the measured range return the boundary
value rather than extrapolating.

In [ ]:
"""
    linear_interp(x, xp, fp) -> Float64

Clamp-at-edges linear interpolation. Queries outside [xp[1], xp[end]]
return the boundary value (no extrapolation).
"""
function linear_interp(x::Real, xp::AbstractVector, fp::AbstractVector)
    x = Float64(x)
    if x <= xp[1]
        return Float64(fp[1])
    elseif x >= xp[end]
        return Float64(fp[end])
    end
    i = searchsortedlast(xp, x)
    x0, x1 = Float64(xp[i]), Float64(xp[i+1])
    f0, f1 = Float64(fp[i]), Float64(fp[i+1])
    return f0 + (f1 - f0) * (x - x0) / (x1 - x0)
end

## 4. Data loading

### Where the files live

The next cell sets `DATA_DIR` to `~/Downloads/`. Override it here if
your files are elsewhere.

In [ ]:
# Look for the data files in `./data/` (when the notebook lives in a
# checked-out repo), then in `../data/`, then in `~/Downloads/`.
# Override DATA_DIR manually if your files are elsewhere.
function _find_data_dir(filename)
    for candidate in (joinpath(pwd(), "data"),
                      joinpath(dirname(pwd()), "data"),
                      joinpath(homedir(), "Downloads"))
        isfile(joinpath(candidate, filename)) && return candidate
    end
    return joinpath(homedir(), "Downloads")  # fallback for error message
end

const DATA_DIR  = _find_data_dir("WT_unstressed_readspermillionreads.csv")
const EXPR_FILE = joinpath(DATA_DIR, "WT_unstressed_readspermillionreads.csv")
const NET_FILE  = joinpath(DATA_DIR, "TF_von_Teufel.csv")

# Sanity check: fail early with a clear message if the files aren't there.
for f in (EXPR_FILE, NET_FILE)
    isfile(f) || error("File not found: $f\n" *
                       "Expected location: $DATA_DIR/\n" *
                       "Either put the file there, or edit DATA_DIR above.")
end
println("Data files found in: $DATA_DIR")


### Expression loader

Reads the CSV, identifies the time columns (those whose header
parses as a number), applies the `max_nans` filter, linearly
interpolates the remaining short gaps, and rescales RPM to estimated
molecules per cell.

In [ ]:
"""
    load_rna_seq(path; max_nans=4) -> (time_axis, gene_names, expression)

Load and clean the RPM matrix. Returns:
  time_axis  :: Vector{Float64}    minutes
  gene_names :: Vector{String}     row labels (length G)
  expression :: Matrix{Float64}    G × T, in molecules per cell

Genes with more than `max_nans` missing timepoints (out of 22) are
dropped. Remaining NaN gaps are filled by linear interpolation along
the time axis. `max_nans = 0` means strict mode (drop any gene with
any NaN).
"""
function load_rna_seq(path::String; max_nans::Int = DEFAULT_MAX_NANS)
    # Read CSV. Empty cells -> missing; CSV.jl auto-infers numeric types.
    df = CSV.read(path, DataFrame)

    # Identify time columns: those whose name parses as a number.
    time_cols = String[]
    time_axis_raw = Float64[]
    for c in names(df)
        v = tryparse(Float64, string(c))
        if v !== nothing
            push!(time_cols, string(c))
            push!(time_axis_raw, v)
        end
    end
    ord = sortperm(time_axis_raw)
    time_cols = time_cols[ord]
    time_axis = time_axis_raw[ord]

    # Helper: detect both `missing` and floating NaN.
    is_missing(v) = ismissing(v) || (v isa Number && isnan(v))

    # Drop rows exceeding the NaN budget.
    keep = trues(nrow(df))
    for i in 1:nrow(df)
        nan_count = 0
        for c in time_cols
            is_missing(df[i, c]) && (nan_count += 1)
        end
        nan_count > max_nans && (keep[i] = false)
    end
    df = df[keep, :]

    # Clean gene names — strip whitespace, drop empty / "nan" / missing.
    # CSV.jl may give us InlineString/String15 for the name column;
    # coerce to String here so downstream code sees one type.
    df[!, :name] = String[ismissing(x) ? "" : String(strip(string(x))) for x in df.name]
    df = filter(row -> !isempty(row.name) && lowercase(row.name) != "nan", df)

    # Build the numeric matrix.
    gene_names = Vector{String}(df.name)
    T = length(time_cols)
    G = nrow(df)
    expression = Matrix{Float64}(undef, G, T)
    for j in 1:T
        col = df[!, time_cols[j]]
        for i in 1:G
            v = col[i]
            expression[i, j] = is_missing(v) ? NaN : Float64(v)
        end
    end

    # Linearly interpolate remaining NaN gaps. Drop any row whose
    # measurements are all NaN (shouldn't happen after the max_nans
    # filter unless max_nans >= number of timepoints).
    keep_after = trues(G)
    for i in 1:G
        row = view(expression, i, :)
        nan_mask = isnan.(row)
        any(nan_mask) || continue
        good = .!nan_mask
        if !any(good)
            keep_after[i] = false
            continue
        end
        x_good = time_axis[good]
        y_good = Float64.(expression[i, good])
        for k in findall(nan_mask)
            expression[i, k] = linear_interp(time_axis[k], x_good, y_good)
        end
    end
    expression = expression[keep_after, :]
    gene_names = gene_names[keep_after]

    # RPM -> estimated molecules per cell.
    expression .*= RPM_TO_MOLECULES
    return time_axis, gene_names, expression
end

### Network loader

The CSV is `tf, relation, substrate`. Numeric strings in the TF column
(`"1.0"`, `"2.0"`, etc., a known parsing artefact in the source file)
are filtered out as junk.

In [ ]:
"""
    load_tf_network(csv_path) -> Vector{Tuple{String,String}}

Read the TF -> substrate edge list. Returns a vector of (TF, substrate)
tuples. Filters out empty rows and numeric-looking TF/substrate names.
"""
function load_tf_network(csv_path::String)
    df = CSV.read(csv_path, DataFrame)
    edges = Tuple{String,String}[]
    for row in eachrow(df)
        tf_val = row[1]
        sub_val = row[3]
        (ismissing(tf_val) || ismissing(sub_val)) && continue
        tf  = strip(string(tf_val))
        sub = strip(string(sub_val))
        (isempty(tf) || isempty(sub)) && continue
        # Filter parsing junk (numeric strings).
        tryparse(Float64, tf)  !== nothing && continue
        tryparse(Float64, sub) !== nothing && continue
        push!(edges, (String(tf), String(sub)))
    end
    return edges
end

"""
    build_regulators(edges, measured) -> Dict{String, Vector{String}}

For each substrate that's in `measured`, list TFs that regulate it AND
are themselves in `measured`. Deduplicates and sorts TF lists.
"""
function build_regulators(edges::Vector{Tuple{String,String}},
                          measured::AbstractSet{String})
    reg = Dict{String,Vector{String}}()
    for (tf, sub) in edges
        if tf in measured && sub in measured
            push!(get!(reg, sub, String[]), tf)
        end
    end
    for k in keys(reg)
        reg[k] = sort(unique(reg[k]))
    end
    return reg
end

## 5. The optimization

> **Reading this section.** This is the densest math in the notebook. If
> you've never seen a linear program before, the [glossary](#glossary)
> at the bottom has one-line definitions of LP, L1 norm, and decision
> variables. The short version: for each gene, we set up a small
> optimization problem that finds the coupling coefficients ($\alpha$)
> that best explain that gene's mRNA trajectory as a weighted sum of
> its regulators' trajectories. We use L1 in two places: as the loss
> (to be robust to outliers) and as a penalty on $\alpha$ (to encourage
> most coefficients to be exactly zero, reflecting that each gene is
> regulated by only a handful of TFs, not all 150).

For each substrate $i$ with regulator set $R(i) = \{j_1, \dots, j_k\}$
and fit timepoints $\{t_1, \dots, t_T\}$:

### Decision variables

- $\alpha_{ij}$ for $j \in R(i)$, the coupling strength. Split into
  $\alpha_{ij} = a^+_{ij} - a^-_{ij}$ with both non-negative, so the L1
  penalty on $|\alpha|$ stays linear in the decision variables.
- $f^+_i(t)$, $f^-_i(t)$ for each $t$, non-negative residual slacks.

### Constraints

For each timepoint:

$$\sum_{j \in R(i)} \alpha_{ij} \cdot TF_j(t - \tau) \;+\; f^+_i(t) - f^-_i(t) \;=\; S_i(t)$$

### Objective

$$\min \;\sum_t \left( f^+_i(t) + f^-_i(t) \right) \;+\; w_2 \,\sum_j \left( a^+_{ij} + a^-_{ij} \right)$$

That is L1 in the residual + L1 in $\alpha$. Convex, sparse, signed,
fast. One HiGHS LP per substrate.

### Time-delay handling

We linearly interpolate the TF series at $t - \tau$. The fit set is
the subset of timepoints where $t - \tau$ lies inside the measured
range, so the first 4 timepoints (t = 0, 5, 10, 15) drop out. That
leaves 18 fit timepoints out of 22.

In [ ]:
"""
    fit_substrate(sub_series, tf_matrix, w2) -> (alpha, residual_l1, converged)

Solve the LP for one substrate. Inputs:

  sub_series :: length-T vector, S(t) at the fit timepoints
  tf_matrix  :: k × T matrix, TF_j(t - tau) for each (j, t)
  w2         :: L1 weight on alpha

Returns:
  alpha       :: length-k vector
  residual_l1 :: sum of f+ + f- at the optimum
  converged   :: Bool
"""
function fit_substrate(sub_series::AbstractVector{Float64},
                       tf_matrix::AbstractMatrix{Float64},
                       w2::Float64)
    k, T = size(tf_matrix)
    @assert length(sub_series) == T "size mismatch in fit_substrate"

    m = Model(HiGHS.Optimizer)
    set_silent(m)

    @variable(m, a_plus[1:k]  >= 0)
    @variable(m, a_minus[1:k] >= 0)
    @variable(m, f_plus[1:T]  >= 0)
    @variable(m, f_minus[1:T] >= 0)

    @constraint(m, [t = 1:T],
        sum(tf_matrix[j, t] * (a_plus[j] - a_minus[j]) for j in 1:k)
        + f_plus[t] - f_minus[t] == sub_series[t])

    @objective(m, Min,
        sum(f_plus[t] + f_minus[t] for t in 1:T)
        + w2 * sum(a_plus[j] + a_minus[j] for j in 1:k))

    optimize!(m)
    # JuMP re-exports MOI's status enums; OPTIMAL is the success case.
    converged = termination_status(m) == OPTIMAL
    if !converged
        return zeros(k), Inf, false
    end
    alpha       = [value(a_plus[j]) - value(a_minus[j]) for j in 1:k]
    residual_l1 = sum(value(f_plus[t]) + value(f_minus[t]) for t in 1:T)
    return alpha, residual_l1, true
end

"""
    fit_all(time_axis, gene_index, expression, regulators; tau, w2)
        -> Vector{NamedTuple}

Fit every substrate with at least one measured regulator. Returns a
vector of NamedTuples (substrate, regulators, alpha, residual_l1,
converged).
"""
function fit_all(time_axis::Vector{Float64},
                 gene_index::Dict{String,Int},
                 expression::Matrix{Float64},
                 regulators::Dict{String,Vector{String}};
                 tau::Float64 = DEFAULT_TAU,
                 w2::Float64  = DEFAULT_W2)
    fit_mask = time_axis .- tau .>= time_axis[1]
    fit_times = time_axis[fit_mask]
    T_eff = length(fit_times)

    results = NamedTuple[]
    for sub in sort(collect(keys(regulators)))
        regs = regulators[sub]
        k = length(regs)
        k == 0 && continue

        sub_series = expression[gene_index[sub], fit_mask]
        tf_matrix = Matrix{Float64}(undef, k, T_eff)
        for (j, tf) in enumerate(regs)
            tf_full = view(expression, gene_index[tf], :)
            for ti in 1:T_eff
                tf_matrix[j, ti] = linear_interp(
                    fit_times[ti] - tau, time_axis, tf_full)
            end
        end
        alpha, resid, ok = fit_substrate(sub_series, tf_matrix, w2)
        push!(results, (substrate = sub,
                        regulators = regs,
                        alpha = alpha,
                        residual_l1 = resid,
                        converged = ok))
    end
    return results
end

## 6. Run it

Loads data with `max_nans = 4`, fits all substrates with $\tau = 20$
min and $w_2 = 5$, prints a summary, and writes a sparse edge-list
CSV to your `~/Downloads/` folder.

Expected runtime on a laptop: a few seconds to load + a few seconds to
fit.

**Expected output (from the verified Python reference):**

```
Genes kept (max_nans=4):       4948
Substrates with regulators:    2406
Fit timepoints (after delay):  18

  5.234 s  (varies on your machine)

Substrates fit:        2406
Converged:             2406
Total alpha edges:     5609
Non-zero alpha edges:  3929
|alpha| median:        0.644
|alpha| max:           78.82
Residual L1 median:    17.55
```

Your numbers should match these to floating-point precision.

In [ ]:
# Run.
time_axis, gene_names, expression = load_rna_seq(EXPR_FILE; max_nans = 4)
gene_index = Dict(g => i for (i, g) in enumerate(gene_names))

edges       = load_tf_network(NET_FILE)
regulators  = build_regulators(edges, Set(gene_names))

println("Genes kept (max_nans=4):       ", length(gene_names))
println("Substrates with regulators:    ", length(regulators))
println("Fit timepoints (after delay):  ",
        sum(time_axis .- DEFAULT_TAU .>= time_axis[1]))
println()

@time results = fit_all(time_axis, gene_index, expression, regulators;
                        tau = DEFAULT_TAU, w2 = DEFAULT_W2)

# Quick stats.
n_conv     = count(r.converged for r in results)
all_alpha  = vcat([r.alpha for r in results]...)
nz         = filter(x -> abs(x) > 1e-9, all_alpha)

println()
println("Substrates fit:        ", length(results))
println("Converged:             ", n_conv)
println("Total alpha edges:     ", length(all_alpha))
println("Non-zero alpha edges:  ", length(nz))
println("|alpha| median:        ", round(median(abs.(nz)); digits = 3))
println("|alpha| max:           ", round(maximum(abs.(nz));  digits = 2))
println("Residual L1 median:    ",
        round(median([r.residual_l1 for r in results]); digits = 2))

# Save the sparse edge-list to ~/Downloads/.
output_csv = joinpath(DATA_DIR, "alpha_w2=5.0_tau=20.0.csv")
open(output_csv, "w") do fh
    println(fh, "substrate,tf,alpha,n_regulators,residual_l1")
    for r in results
        for (tf, a) in zip(r.regulators, r.alpha)
            if abs(a) >= 1e-9
                println(fh, r.substrate, ",", tf, ",",
                        @sprintf("%.6f", a), ",",
                        length(r.regulators), ",",
                        @sprintf("%.4f", r.residual_l1))
            end
        end
    end
end
println("\nWrote ", output_csv)

## 7. Validation: cell-cycle canaries

The strongest sanity check: does the fit identify textbook regulators?
Six canary genes whose regulators are well established:

| Gene | Function | Textbook regulator |
|------|----------|--------------------|
| CLN2 | G1 cyclin | SBF (Swi4 / Swi6) |
| ACE2 | M / G1 TF | Fkh2 / Mcm1 |
| CLB5 | S-phase cyclin | MBF (Mbp1 / Swi6) |
| CLN3 | upstream G1 cyclin | mostly an input — less clean expectation |
| SIC1 | Cdk1 inhibitor at M / G1 | Ace2 / Swi5 |
| CDC20 | APC/C activator at anaphase | Mcm1 / Fkh2 |

Each panel: blue circles are observed mRNA, red squares are the fitted
linear model's prediction. The title shows the top non-zero fitted
$\alpha$ values.

The reference figure for this panel is in
`figures_reference/03_canaries_max_nans_4.png`. Your Julia plot should
match it visually.

In [ ]:
# Six well-known cell-cycle genes. Tests whether the fit recovers
# textbook regulators.
const CANARIES = ["CLN2", "ACE2", "CLB5", "CLN3", "SIC1", "CDC20"]

fit_mask  = time_axis .- DEFAULT_TAU .>= time_axis[1]
fit_times = time_axis[fit_mask]

plts = []
for sub in CANARIES
    if !haskey(regulators, sub)
        push!(plts, plot(title = "$sub  —  not in dataset"))
        continue
    end
    regs = regulators[sub]
    sub_series = expression[gene_index[sub], fit_mask]
    tf_matrix = Matrix{Float64}(undef, length(regs), length(fit_times))
    for (j, tf) in enumerate(regs)
        tf_full = view(expression, gene_index[tf], :)
        for ti in eachindex(fit_times)
            tf_matrix[j, ti] = linear_interp(
                fit_times[ti] - DEFAULT_TAU, time_axis, tf_full)
        end
    end
    alpha, _, _ = fit_substrate(sub_series, tf_matrix, DEFAULT_W2)
    pred = vec(alpha' * tf_matrix)

    full_S = expression[gene_index[sub], :]
    nz_pairs = sort([(r, a) for (r, a) in zip(regs, alpha) if abs(a) > 1e-9];
                    by = x -> -abs(x[2]))
    title_str = sub * "  " *
        join([string(r, "=", round(a; digits=2))
              for (r, a) in nz_pairs[1:min(end, 3)]], ", ")

    p = plot(time_axis, full_S, label = "observed", marker = :circle,
             color = :steelblue, lw = 2, ms = 4,
             xlabel = "time (min)", ylabel = "mRNA (molecules / cell)",
             title = title_str, titlefont = font(9), legend = :topright,
             grid = true)
    plot!(p, fit_times, pred, label = "predicted", marker = :square,
          color = :firebrick, lw = 2, ms = 4, ls = :dash)
    push!(plts, p)
end
plot(plts..., layout = (2, 3), size = (1300, 750),
     left_margin   = 5Plots.mm,
     right_margin  = 3Plots.mm,
     bottom_margin = 4Plots.mm,
     top_margin    = 3Plots.mm)

### Reading the panel

- **CLN2 → SWI4 = +1.38** (dominant). SBF activating CLN2 in G1.
  STE12 also picks up a +0.53, which is mating-pathway cross-talk
  rather than the primary cell-cycle regulator.
- **ACE2 → FKH2 = +0.56** (dominant). Textbook Fkh2 / Mcm1 contribution
  to late-M / G1 expression.
- **CLB5 → SWI6 = +0.35, SWI4 = +0.20, MBP1 = +0.05**. All three
  SBF/MBF components positive, consistent with S-phase regulation.
- **CLN3** — less clean. MCM1 and RAP1 positive, SWI4 *negative*. CLN3
  is more an input to the cell-cycle TFs than an output of them, so
  a linear model with these regulators struggles here. This is a
  known weak spot of the linear formulation.
- **SIC1 → ACE2 = +1.37** (dominant). Ace2 is the canonical SIC1
  activator at M / G1 exit. Textbook.
- **CDC20** — PDR1 and ABF1 dominant. Less canonical; the textbook
  Mcm1 / Fkh2 regulators are in the network but with smaller $\alpha$
  here. Possibly an artefact of which regulators are listed in this
  particular network curation.

### The amplitude problem

Most panels share the same pattern: the model captures the *qualitative*
oscillation but underestimates the peak amplitude. CLN2 peaks at ~25
molecules per cell but the model only reaches ~8. Reasons (any of):

1. The network is missing an important regulator (CLN2 is missing
   MBP1 in this network, for example).
2. The linear formulation can't reach high peaks when the available
   regulators don't oscillate strongly themselves.
3. Active TF protein, not TF mRNA, is the real driver — there are
   post-transcriptional layers we're not capturing.

This is the strongest argument for one of: extending the network,
adding a multiplicative non-linear formulation, or including
protein-level data.

## 8. Why NaN handling matters

The Teufel 2019 matrix has scattered NaN values. Of the 6,701 rows,
4,370 have no NaN at all, 941 are entirely empty, and the rest are
in between.

The full coverage table:

| `max_nans`    | Genes kept | TFs available (of 147 real) |
|---------------|------------|------------------------------|
| `0` (strict)  | 4,370      | 90                           |
| `1`           | 4,649      | 104                          |
| `2`           | 4,797      | 111                          |
| `4` (default) | 4,948      | 117                          |
| `8`           | 5,132      | 126                          |
| `22`          | 6,651      | 147                          |

Only **5 TFs are truly unmeasured anywhere** in the dataset
(GAT3, HMS1, INO2, RGM1, RTG1). Everything else is recoverable with
enough interpolation tolerance.

**Strict filtering drops the canonical cell-cycle TFs**:

| TF | NaNs | Dropped at strict? |
|------|------|----|
| SWI4 | 2 | yes |
| FKH1 | 1 | yes |
| SWI5 | 4 | yes (boundary) |
| MSN4 | 5 | yes |
| HAP4 | 5 | yes |
| NDD1 | 11 | yes |
| GAL4 | 18 | yes |

while these are fully measured:

| TF | NaNs |
|------|------|
| MBP1, SWI6, ACE2, STE12, FKH2 | 0 |

So strict filtering means the LP fitting CLN2 sees STE12 (mating
pathway, fully measured) but not SWI4 (the real G1 cyclin regulator,
2 NaNs). It picks STE12 as dominant because it has no other choice.
With `max_nans = 4`, SWI4 is interpolated through its 2 missing
timepoints and the LP correctly identifies it as dominant.

The figure below shows CLN2 fit twice: strict vs relaxed. Reference
figure in `figures_reference/04_cln2_strict_vs_relaxed.png`.

### NaN distribution and TF coverage

Before showing the strict-vs-relaxed CLN2 fit, here are the two
plots that justify the choice of `max_nans = 4`:

1. **The NaN distribution** is bimodal. Most genes are either fully
   measured (4,370 with zero NaN) or entirely empty (941 with all 22
   NaN). The 1,390 genes in between mostly have a handful of gaps.
2. **TF coverage** climbs sharply between `max_nans = 0` and
   `max_nans = 4`, then flattens. By 4 we have 117 of 147 real TFs;
   pushing higher gains diminishing returns.

In [ ]:
# NaN distribution and TF coverage plots.
# We reload the raw CSV WITHOUT filtering so we can see the full
# distribution.
raw_df = CSV.read(EXPR_FILE, DataFrame)
raw_time_cols = String[c for c in names(raw_df) if tryparse(Float64, string(c)) !== nothing]
raw_nan_counts = Int[]
for i in 1:nrow(raw_df)
    n = 0
    for c in raw_time_cols
        v = raw_df[i, c]
        if ismissing(v) || (v isa Number && isnan(v))
            n += 1
        end
    end
    push!(raw_nan_counts, n)
end

# Set of TFs in the network (filter junk).
all_tfs_in_net = Set{String}()
for (tf, _) in edges
    push!(all_tfs_in_net, tf)
end

# Build name -> NaN-count map (for raw data).
raw_name_to_nans = Dict{String,Int}()
for i in 1:nrow(raw_df)
    nm = raw_df[i, :name]
    ismissing(nm) && continue
    nm_str = String(strip(string(nm)))
    isempty(nm_str) && continue
    raw_name_to_nans[nm_str] = raw_nan_counts[i]
end

# Coverage curve: at each max_nans threshold, how many TFs and total
# genes are kept?
thresholds = 0:22
tfs_avail = Int[]
genes_kept = Int[]
for thr in thresholds
    avail = count(tf -> get(raw_name_to_nans, tf, 99) <= thr, all_tfs_in_net)
    kept  = count(n -> n <= thr, raw_nan_counts)
    push!(tfs_avail, avail)
    push!(genes_kept, kept)
end

# Plot 1: NaN distribution histogram.
p_nan = histogram(raw_nan_counts, bins = -0.5:1:22.5,
                  color = :steelblue, linecolor = :white,
                  label = "",   # suppress default "y1" series label
                  xlabel = "Number of NaN timepoints per gene (of 22)",
                  ylabel = "Number of genes",
                  title = "NaN distribution across the Teufel 2019 RNA-seq matrix\n(n=$(nrow(raw_df)) rows total)",
                  titlefont = font(9), legend = :topright, grid = true)
vline!(p_nan, [0.5],  ls = :dash, lw = 1.5, color = :red,
       label = "max_nans=0 (strict)")
vline!(p_nan, [4.5],  ls = :dash, lw = 1.5, color = :orange,
       label = "max_nans=4 (default)")
vline!(p_nan, [8.5],  ls = :dash, lw = 1.5, color = :green,
       label = "max_nans=8")

# Plot 2: TF coverage and total gene count vs threshold.
p_cov = plot(thresholds, tfs_avail,
             marker = :circle, lw = 2, color = :steelblue,
             label = "TFs available",
             xlabel = "max_nans threshold",
             ylabel = "TFs with usable time course",
             title = "TF coverage vs NaN tolerance",
             titlefont = font(9), grid = true, legend = :bottomright)
vline!(p_cov, [0], ls = :dash, color = :red,    label = "")
vline!(p_cov, [4], ls = :dash, color = :orange, label = "")
plot(p_nan, p_cov, layout = (1, 2), size = (1400, 500),
     left_margin   = 6Plots.mm,
     right_margin  = 4Plots.mm,
     bottom_margin = 6Plots.mm,
     top_margin    = 4Plots.mm)

### CLN2 with strict filtering vs `max_nans = 4`

In [ ]:
# CLN2 fit twice: max_nans=0 (strict) and max_nans=4 (recommended).
# We reload the data at each setting to be honest about what changes.
function fit_cln2(max_nans::Int)
    t_axis, g_names, expr = load_rna_seq(EXPR_FILE; max_nans = max_nans)
    g_idx = Dict(g => i for (i, g) in enumerate(g_names))
    regs_full = build_regulators(edges, Set(g_names))
    haskey(regs_full, "CLN2") || return nothing
    regs = regs_full["CLN2"]
    fmask = t_axis .- DEFAULT_TAU .>= t_axis[1]
    ftimes = t_axis[fmask]
    S_fit = expr[g_idx["CLN2"], fmask]
    TF = Matrix{Float64}(undef, length(regs), length(ftimes))
    for (j, tf) in enumerate(regs)
        tf_full = view(expr, g_idx[tf], :)
        for ti in eachindex(ftimes)
            TF[j, ti] = linear_interp(ftimes[ti] - DEFAULT_TAU, t_axis, tf_full)
        end
    end
    alpha, _, _ = fit_substrate(S_fit, TF, DEFAULT_W2)
    return (t_axis = t_axis,
            S_full = expr[g_idx["CLN2"], :],
            ftimes = ftimes,
            pred   = vec(alpha' * TF),
            regs   = regs,
            alpha  = alpha)
end

panels = []
for (max_nans, label) in [(0, "strict (max_nans=0)"),
                          (4, "recommended (max_nans=4)")]
    r = fit_cln2(max_nans)
    nz_pairs = sort([(t, a) for (t, a) in zip(r.regs, r.alpha) if abs(a) > 1e-9];
                    by = x -> -abs(x[2]))
    top = join([string(t, "=", round(a; digits=2)) for (t, a) in nz_pairs[1:min(end, 3)]], ", ")
    p = plot(r.t_axis, r.S_full, label = "observed", marker = :circle,
             color = :steelblue, lw = 2, ms = 4,
             xlabel = "time (min)", ylabel = "mRNA (molec / cell)",
             title = "CLN2 — $label\n$top",
             titlefont = font(9), legend = :topright, grid = true)
    plot!(p, r.ftimes, r.pred, label = "predicted", marker = :square,
          color = :firebrick, lw = 2, ms = 4, ls = :dash)
    push!(panels, p)
end
plot(panels..., layout = (1, 2), size = (1300, 500),
     left_margin   = 6Plots.mm,
     right_margin  = 4Plots.mm,
     bottom_margin = 6Plots.mm,
     top_margin    = 4Plots.mm)

## 9. Regularization sweep — the L-curve

The L1 penalty weight $w_2$ controls the sparsity vs fit trade-off.
We sweep over decades and look at the non-zero $\alpha$ count, median
residual, and max $|\alpha|$.

**Expected output (from the verified Python reference):**

```
w2=  0.01: nz= 5603  res_med= 16.73  |a|_med=0.6868  |a|_max=1083.25
w2=   0.1: nz= 5546  res_med= 16.74  |a|_med=0.6773  |a|_max= 166.13
w2=     1: nz= 4970  res_med= 16.88  |a|_med=0.6481  |a|_max= 134.60
w2=     5: nz= 3929  res_med= 17.55  |a|_med=0.6438  |a|_max=  78.82
w2=    10: nz= 3464  res_med= 18.43  |a|_med=0.6427  |a|_max=  82.39
w2=    50: nz= 1655  res_med= 37.30  |a|_med=0.4770  |a|_max= 116.08
w2=   100: nz=  470  res_med= 56.57  |a|_med=0.1391  |a|_max=  60.61
```

The right-panel L-curve has its elbow between $w_2 = 5$ and $w_2 = 50$.
We default to $w_2 = 5$ because (a) it sits at the elbow on the
residual side, (b) it brings $|\alpha|_\max$ down to a sensible
$\sim 80$ from the unregularized $\sim 1000$, and (c) it preserves
the canonical regulators in the canaries above. Reference figure in
`figures_reference/05_lcurve.png`.

In [ ]:
# L-curve sweep over w2.
const W2_GRID = [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
sweep_results = NamedTuple[]
for w2 in W2_GRID
    r = fit_all(time_axis, gene_index, expression, regulators;
                tau = DEFAULT_TAU, w2 = w2)
    all_a = vcat([x.alpha for x in r]...)
    nz_a  = filter(x -> abs(x) > 1e-9, all_a)
    push!(sweep_results, (
        w2      = w2,
        n_nz    = length(nz_a),
        a_med   = isempty(nz_a) ? 0.0 : median(abs.(nz_a)),
        a_max   = isempty(nz_a) ? 0.0 : maximum(abs.(nz_a)),
        res_med = median([x.residual_l1 for x in r]),
    ))
    s = sweep_results[end]
    @printf("w2=%6g: nz=%5d  res_med=%6.2f  |a|_med=%6.4f  |a|_max=%7.2f\n",
            w2, s.n_nz, s.res_med, s.a_med, s.a_max)
end

p1 = plot(W2_GRID, [s.n_nz for s in sweep_results],
          xscale = :log10, marker = :circle, lw = 2,
          color = :steelblue,
          xlabel = "w2 (regularization weight)",
          ylabel = "Non-zero alpha count",
          title = "Sparsity vs regularization",
          grid = true, legend = false)

p2 = plot([s.res_med for s in sweep_results],
          [s.a_max   for s in sweep_results],
          xscale = :log10, yscale = :log10,
          marker = :circle, lw = 2, color = :steelblue,
          xlabel = "Median residual L1",
          ylabel = "Max |alpha|",
          title = "L-curve",
          grid = true, legend = false)
# Only annotate the endpoints + default w2=5, otherwise points bunch
# in the elbow. Other w2 values stay as markers without labels.
annotate_set = Set([0.01, 0.1, 5.0, 50.0, 100.0])
for s in sweep_results
    if s.w2 in annotate_set
        # Slight vertical offset to keep label from sitting on the marker.
        offset_factor = s.w2 == 5.0 ? 0.85 : 1.15
        annotate!(p2, s.res_med * 1.06, s.a_max * offset_factor,
                  text("w2=" * string(s.w2), :left, 7))
    end
end
plot(p1, p2, layout = (1, 2), size = (1200, 450),
     left_margin   = 6Plots.mm,
     right_margin  = 6Plots.mm,
     bottom_margin = 6Plots.mm,
     top_margin    = 4Plots.mm)

## 10. Global fit quality

A few diagnostics that summarize the ~2,400 per-substrate fits.

**Left panel** — distribution of $\log_{10}|\alpha|$. Most non-zero
coefficients sit between $10^{-1}$ and $10^{1}$ (median $|\alpha|$ ~
0.6); a long tail extends to ~80. The single huge $|\alpha|_\max$ at
$w_2 = 0.01$ in section 9 is exactly the kind of vertex solution the
L1 penalty exists to suppress.

**Middle panel** — residual $L_1$ scales with the number of
regulators. Substrates with more TFs have more degrees of freedom
and fit tighter, as expected.

**Right panel** — per-substrate $R^2$ distribution. **Note the
median is negative (around $-0.9$) and only a small fraction of
substrates clear $R^2 > 0.5$.** This is a real and honest finding
about the doc-literal formulation: without an intercept, the model
is forced through the origin per timepoint, which means a substrate
whose mean level is decoupled from its regulators' mean levels gets
penalized hard on absolute residuals. The L1 fit loss is robust to
this in absolute terms (residuals stay manageable), but R² is a
ratio against the variance of $S_i$ alone, so it punishes the
forced-through-origin baseline.

The biology still comes out right: the *direction* and *relative
magnitudes* of $\alpha$ identify the canonical regulators
(CLN2 → SWI4, SIC1 → ACE2, etc., section 7). The fits underestimate
peak amplitudes (section 7 commentary), and the residuals are
dominated by that amplitude shortfall plus the missing-intercept
baseline offset. Adding an intercept term recovers $R^2$ but at
$w_2 = 5$ collapses the biological $\alpha$'s to zero — see the
**Design decisions** table in section 12. Resolving this properly is
a v2 question.

For now: $R^2$ is the wrong metric to lead with for this model;
$\alpha$ direction-matching against textbook biology is the right
one, and on that test the model passes for the cell-cycle genes.

In [ ]:
# Build per-substrate fit-quality stats from the existing `results`
# object (set in section 6 at the defaults tau=20, w2=5).
fit_mask  = time_axis .- DEFAULT_TAU .>= time_axis[1]
fit_times = time_axis[fit_mask]

all_nz_alphas = Float64[]
n_regs_per_sub = Int[]
resid_per_sub  = Float64[]
r2_per_sub     = Float64[]

for r in results
    r.converged || continue
    # Recompute the prediction the same way as in fit_all.
    k = length(r.regulators)
    TF = Matrix{Float64}(undef, k, length(fit_times))
    for (j, tf) in enumerate(r.regulators)
        tf_full = view(expression, gene_index[tf], :)
        for ti in eachindex(fit_times)
            TF[j, ti] = linear_interp(fit_times[ti] - DEFAULT_TAU,
                                      time_axis, tf_full)
        end
    end
    S    = expression[gene_index[r.substrate], fit_mask]
    pred = vec(r.alpha' * TF)
    ss_res = sum((S .- pred) .^ 2)
    ss_tot = sum((S .- mean(S)) .^ 2)
    r2     = ss_tot > 0 ? 1 - ss_res / ss_tot : NaN

    push!(n_regs_per_sub, k)
    push!(resid_per_sub, r.residual_l1)
    push!(r2_per_sub, r2)
    for a in r.alpha
        abs(a) > 1e-9 && push!(all_nz_alphas, a)
    end
end

# Panel 1: histogram of log10(|alpha|) on a LINEAR x-axis.
# (Plots.jl's xscale=:log10 with histogram is fragile on Julia 1.8
# + older Plots versions and produces "no strict ticks" warnings
# plus an unrenderable plot. Transforming the data instead works.)
abs_alphas = abs.(all_nz_alphas)
log_alphas = log10.(filter(x -> x > 0, abs_alphas))
p_alpha = histogram(log_alphas, bins = 40,
                    color = :steelblue, linecolor = :white,
                    xlabel = "log10(|alpha|)",
                    ylabel = "Count",
                    title = "Distribution of fitted |alpha|",
                    titlefont = font(9), legend = false, grid = true)

# Panel 2: residual vs n_regulators.
p_resid = scatter(n_regs_per_sub, resid_per_sub,
                  ms = 2, alpha = 0.3, color = :steelblue,
                  xlabel = "# regulators per substrate",
                  ylabel = "Residual L1",
                  title = "Fit residual vs regulator count",
                  titlefont = font(9), legend = false, grid = true)

# Panel 3: per-substrate R^2 distribution.
finite_r2 = filter(isfinite, r2_per_sub)
r2_clamped = clamp.(finite_r2, -2.0, 1.0)
med_r2 = round(median(finite_r2); digits = 3)
frac_above = round(100 * mean(finite_r2 .> 0.5); digits = 1)
p_r2 = histogram(r2_clamped, bins = 40,
                 color = :steelblue, linecolor = :white,
                 xlabel = "R^2 (clamped to [-2, 1])",
                 ylabel = "# substrates",
                 title = "Fit quality (R^2): median=$med_r2, $(frac_above)% > 0.5",
                 titlefont = font(9), legend = false, grid = true)
vline!(p_r2, [0.0], color = :black, lw = 1)
vline!(p_r2, [median(finite_r2)], color = :red, ls = :dash, lw = 1.5)

plot(p_alpha, p_resid, p_r2,
     layout = (1, 3), size = (1600, 500),
     left_margin   = 6Plots.mm,
     right_margin  = 4Plots.mm,
     bottom_margin = 6Plots.mm,
     top_margin    = 4Plots.mm)

## 11. Inspect a single substrate

Use this cell to dive into one gene. Change the argument to inspect
others. Useful candidates beyond the six canaries: CLB6 (S-phase),
HTA1 / HTB1 (histones, peak at S), CLB1 / CLB2 (mitotic cyclins, peak
at M).

In [ ]:
"""
    inspect_substrate(sub; tau=20, w2=5)

Fit one substrate using the currently-loaded data and plot the result.
"""
function inspect_substrate(sub::String;
                           tau::Float64 = DEFAULT_TAU,
                           w2::Float64  = DEFAULT_W2)
    if !haskey(regulators, sub)
        println("Substrate '", sub, "' has no measured regulators in the network.")
        haskey(gene_index, sub) || println("  (Also not in the filtered RNA-seq matrix.)")
        return nothing
    end
    regs = regulators[sub]
    fmask = time_axis .- tau .>= time_axis[1]
    ftimes = time_axis[fmask]
    S_fit = expression[gene_index[sub], fmask]
    TF = Matrix{Float64}(undef, length(regs), length(ftimes))
    for (j, tf) in enumerate(regs)
        tf_full = view(expression, gene_index[tf], :)
        for ti in eachindex(ftimes)
            TF[j, ti] = linear_interp(ftimes[ti] - tau, time_axis, tf_full)
        end
    end
    alpha, resid, ok = fit_substrate(S_fit, TF, w2)
    pred = vec(alpha' * TF)

    println("Substrate: ", sub)
    println("Regulators (", length(regs), "): ", join(regs, ", "))
    println()
    @printf("Fit at w2=%g, tau=%g: converged=%s, residual_L1=%.4f\n",
            w2, tau, ok, resid)
    println("alpha:")
    for (tf, a) in zip(regs, alpha)
        tag = abs(a) > 1e-9 ? "" : "  (zero)"
        @printf("  %-10s %+.4f%s\n", tf, a, tag)
    end

    full_S = expression[gene_index[sub], :]
    nz_pairs = [(r, a) for (r, a) in zip(regs, alpha) if abs(a) > 1e-9]
    top = join([string(r, "=", round(a; digits=2)) for (r, a) in nz_pairs[1:min(end, 3)]], ", ")
    p = plot(time_axis, full_S, label = "observed", marker = :circle,
             color = :steelblue, lw = 2, ms = 5,
             xlabel = "time (min)", ylabel = "mRNA (molecules / cell)",
             title = sub * "    " * top, legend = :topright, grid = true)
    plot!(p, ftimes, pred, label = "predicted", marker = :square,
          color = :firebrick, lw = 2, ms = 5, ls = :dash)
    return p
end

# Should reproduce SWI4 = +1.38, STE12 = +0.53 at the defaults.
inspect_substrate("CLN2")

## 12. Design decisions, with justifications

| Decision | Choice | Why |
|---|---|---|
| Loss | L1 on residual | Robust to outliers; convex; exact-zero residuals possible. Matches the strategy doc. |
| Regularizer | L1 on $\alpha$ | Strategy doc has $\sum |\alpha_{ij}|$. Gives sparse $\alpha$. |
| Reg weight | $w_2 = 5$ | Empirical L-curve elbow. Inert below ~1; residuals climb above ~50. |
| Delay | $\tau = 20$ min | Per strategy doc. |
| Delay handling | Linear interp, no extrapolation | Drops 4 of 22 timepoints. Honest. |
| NaN handling | `max_nans = 4` + linear interp | Recovers cell-cycle TFs that strict filtering drops. Validated on CLN2 → SWI4. |
| RPM rescaling | × 60,000 / 10⁶ | Per strategy doc. |
| Sign of $\alpha$ | Free | No prior reason to restrict to activation only. |
| Intercept term | **Not included** | Tested separately. At $w_2 = 5$ an intercept absorbs the mean and the L1 penalty zeros out *all* regulators, including the biological ones (CLN2 → SWI4 disappears). Doc-literal is correct here. |
| Degradation | Not included in v1 | Strategy doc has the static formulation. Easy to add in v2. |
| Per-substrate independence | Yes | Each LP solves in ms. Embarrassingly parallel if needed. |
| Solver | HiGHS via JuMP | Free, fast, exact on LPs at this scale. |

## 13. Handoff for the whole-cell model

This section writes a folder `tf_handoff/` to `output/` (in the repo) or to
`~/Downloads/` if there's no writable repo output folder,
plus a zip and an Excel workbook of the same contents. Anyone
extending the larger whole-cell model with a transcription module
should be able to consume the handoff without reading this notebook.

The folder contains:

| File | What it is |
|---|---|
| `README.md` | What this is, units, conventions, a Julia load snippet |
| `metadata.json` | Provenance: fit parameters, source data, timestamp, counts |
| `alpha_matrix.csv` | Dense substrates × TFs matrix. Mostly zero. |
| `alpha_edges.csv` | Long form: substrate, tf, alpha (all candidate edges, including zeros) |
| `tf_network_candidate.csv` | Structural connectivity from Teufel: every (substrate, tf) pair where both are measured. Use to refit α with a different method. |
| `tf_network_fitted.csv` | Post-fit connectivity: only edges with $|\alpha| > 0$. Use if you trust the L1 fit. |

**The candidate vs fitted distinction matters.** The candidate
network is structural, derived from the Teufel paper before any
fitting. The fitted network is the L1 sparsification of it. If
someone wants to refit with Bayesian regression or a different
penalty, they want the candidate network. If they want to use our
fit directly, they want the fitted network.

For emailing, the cell below also writes `tf_handoff.zip` (all
files in one archive) and `tf_handoff.xlsx` (all files as
sheets in one Excel workbook).

In [ ]:
# Build the handoff folder + zip + xlsx under ~/Downloads/.
# Uses the in-memory `results` and `regulators` from sections 4-6.
using Dates
using ZipFile
using XLSX

# Output goes to ./output/ when the notebook lives in a checked-out
# repo (we look for a writable sibling of `data/`), else falls back to
# ~/Downloads/ so the notebook still works standalone.
function _find_output_dir()
    for candidate in (joinpath(pwd(), "output"),
                      joinpath(dirname(pwd()), "output"))
        try
            isdir(candidate) || mkpath(candidate)
            # Test writability
            test_file = joinpath(candidate, ".write_test")
            touch(test_file); rm(test_file)
            return candidate
        catch
            continue
        end
    end
    return joinpath(homedir(), "Downloads")
end

OUTPUT_DIR   = _find_output_dir()
HANDOFF_DIR  = joinpath(OUTPUT_DIR, "tf_handoff")
HANDOFF_ZIP  = joinpath(OUTPUT_DIR, "tf_handoff.zip")
HANDOFF_XLSX = joinpath(OUTPUT_DIR, "tf_handoff.xlsx")
isdir(HANDOFF_DIR) || mkpath(HANDOFF_DIR)

# ---- Canonical ordered lists of substrates and TFs ----
substrate_order = sort([r.substrate for r in results])
tf_set = Set{String}()
for r in results, tf in r.regulators
    push!(tf_set, tf)
end
tf_order = sort(collect(tf_set))

sub_idx = Dict(s => i for (i, s) in enumerate(substrate_order))
tf_idx  = Dict(t => j for (j, t) in enumerate(tf_order))

# ---- Dense alpha matrix ----
n_sub = length(substrate_order)
n_tf  = length(tf_order)
alpha_mat = zeros(Float64, n_sub, n_tf)
for r in results
    i = sub_idx[r.substrate]
    for (tf, a) in zip(r.regulators, r.alpha)
        alpha_mat[i, tf_idx[tf]] = a
    end
end

# ---- Counts ----
n_candidate_edges = sum(length(regulators[s]) for s in substrate_order)
n_fitted_edges    = sum(count(a -> abs(a) > 1e-9, r.alpha) for r in results)

# ===========================================================
# Write the individual CSV files into HANDOFF_DIR
# ===========================================================

# alpha_matrix.csv
open(joinpath(HANDOFF_DIR, "alpha_matrix.csv"), "w") do fh
    print(fh, "substrate")
    for tf in tf_order
        print(fh, ",", tf)
    end
    println(fh)
    for (i, s) in enumerate(substrate_order)
        print(fh, s)
        for j in 1:n_tf
            a = alpha_mat[i, j]
            print(fh, ",", a == 0.0 ? "0" : @sprintf("%.6f", a))
        end
        println(fh)
    end
end

# alpha_edges.csv: long form, all candidate edges (includes zeros)
open(joinpath(HANDOFF_DIR, "alpha_edges.csv"), "w") do fh
    println(fh, "substrate,tf,alpha")
    for r in results, (tf, a) in zip(r.regulators, r.alpha)
        println(fh, r.substrate, ",", tf, ",", @sprintf("%.6f", a))
    end
end

# tf_network_candidate.csv: pre-fit structural network
open(joinpath(HANDOFF_DIR, "tf_network_candidate.csv"), "w") do fh
    println(fh, "substrate,tf")
    for s in substrate_order, tf in regulators[s]
        println(fh, s, ",", tf)
    end
end

# tf_network_fitted.csv: only nonzero alpha edges
open(joinpath(HANDOFF_DIR, "tf_network_fitted.csv"), "w") do fh
    println(fh, "substrate,tf,alpha")
    for r in results, (tf, a) in zip(r.regulators, r.alpha)
        if abs(a) > 1e-9
            println(fh, r.substrate, ",", tf, ",", @sprintf("%.6f", a))
        end
    end
end

# metadata.json
META_JSON_STR = """{
  "schema_version": "1.0",
  "generated_at_utc": "$(Dates.now(Dates.UTC))",
  "model": "S_i(t) = sum_j alpha_ij * TF_j(t - tau)",
  "fit_params": {
    "tau_min": $DEFAULT_TAU,
    "w2_l1_penalty": $DEFAULT_W2,
    "max_nans_per_gene": $DEFAULT_MAX_NANS,
    "molecules_per_cell_reference": $MOLECULES_PER_CELL,
    "solver": "HiGHS via JuMP",
    "loss": "L1 on residuals + L1 penalty on alpha"
  },
  "source_data": {
    "expression": "Teufel et al. 2019 (Scientific Reports 9:3343), WT unstressed RPM",
    "network": "Teufel et al. 2019 supplementary table 4"
  },
  "counts": {
    "n_substrates": $n_sub,
    "n_tfs": $n_tf,
    "n_candidate_edges": $n_candidate_edges,
    "n_fitted_edges_nonzero": $n_fitted_edges,
    "n_timepoints_total": $(length(time_axis)),
    "n_timepoints_used_in_fit": $(sum(time_axis .- DEFAULT_TAU .>= time_axis[1]))
  },
  "units": {
    "alpha": "dimensionless coupling (molecules/cell per molecules/cell)",
    "expression": "molecules per cell",
    "tau": "minutes",
    "time": "minutes since alpha-factor release"
  }
}
"""
open(joinpath(HANDOFF_DIR, "metadata.json"), "w") do fh
    print(fh, META_JSON_STR)
end

# README.md
README_STR = """# Transcription factor coupling handoff

Fitted coefficients for `S_i(t) = sum_j alpha_ij * TF_j(t - tau)` on the
Teufel 2019 *S. cerevisiae* WT unstressed time course. `tau = $(Int(DEFAULT_TAU))` min.
Both `S` and `TF` are in molecules per cell (RPM rescaled by `$MOLECULES_PER_CELL / 1e6`).
The fit is doc-literal: no intercept, no degradation term. See section 10 of the
generating notebook for the intercept trade-off.

## Files

- `metadata.json`: fit parameters, source data, counts, timestamp.
- `alpha_matrix.csv`: dense `n_substrates x n_tfs` matrix, header row gives TF order.
- `alpha_edges.csv`: long form `substrate, tf, alpha`. All candidate edges, including `alpha = 0`.
- `tf_network_candidate.csv`: structural connectivity from Teufel, filtered to measured TFs and substrates. Use to refit alpha with a different method.
- `tf_network_fitted.csv`: only edges with `abs(alpha) > 0` after L1 sparsification. Use if you trust the fit.

## Conventions

- `alpha = 0` means "fit drove it to zero", not "no edge in the network". Absent edges do not appear in any file.
- Sign of alpha is free: positive activates, negative represses.
- Linear (not log) space throughout.

## Load in Julia

```julia
using DataFrames, CSV, JSON3

meta  = JSON3.read(read("metadata.json", String))
alpha = CSV.read("alpha_matrix.csv", DataFrame)
edges = CSV.read("tf_network_fitted.csv", DataFrame)

# alpha[alpha.substrate .== "CLN2", :SWI4]   # about 1.38
```

## Provenance

Generated from `transcription_fit.ipynb` at the timestamp in `metadata.json`.
Source: Teufel et al. 2019, *Scientific Reports* 9:3343,
[doi:10.1038/s41598-019-39850-7](https://doi.org/10.1038/s41598-019-39850-7).
"""
open(joinpath(HANDOFF_DIR, "README.md"), "w") do fh
    print(fh, README_STR)
end

# ===========================================================
# Zip the folder (cross-platform via ZipFile.jl)
# ===========================================================
isfile(HANDOFF_ZIP) && rm(HANDOFF_ZIP)
let zw = ZipFile.Writer(HANDOFF_ZIP)
    for fname in ("README.md", "metadata.json", "alpha_matrix.csv",
                  "alpha_edges.csv", "tf_network_candidate.csv",
                  "tf_network_fitted.csv")
        entry = ZipFile.addfile(zw, fname; method = ZipFile.Deflate)
        write(entry, read(joinpath(HANDOFF_DIR, fname)))
    end
    close(zw)
end

# ===========================================================
# Excel workbook with the same contents as named sheets.
# Uses XLSX.writetable! (column-oriented batch write -- fast).
# ===========================================================

# Prepare each table as (columns, labels).
matrix_cols = Vector{Any}(undef, n_tf + 1)
matrix_cols[1] = substrate_order
for j in 1:n_tf
    matrix_cols[j+1] = alpha_mat[:, j]
end
matrix_labels = vcat(["substrate"], tf_order)

edges_sub = String[]
edges_tf  = String[]
edges_a   = Float64[]
for r in results, (tf, a) in zip(r.regulators, r.alpha)
    push!(edges_sub, r.substrate)
    push!(edges_tf, tf)
    push!(edges_a, a)
end

cand_sub = String[]
cand_tf  = String[]
for s in substrate_order, tf in regulators[s]
    push!(cand_sub, s)
    push!(cand_tf, tf)
end

fit_sub = String[]
fit_tf  = String[]
fit_a   = Float64[]
for r in results, (tf, a) in zip(r.regulators, r.alpha)
    if abs(a) > 1e-9
        push!(fit_sub, r.substrate)
        push!(fit_tf, tf)
        push!(fit_a, a)
    end
end

# Flatten metadata to a 2-column table for the metadata sheet.
meta_keys = ["schema_version", "generated_at_utc", "model",
             "tau_min", "w2_l1_penalty", "max_nans_per_gene",
             "molecules_per_cell_reference", "solver", "loss",
             "source_expression", "source_network",
             "n_substrates", "n_tfs",
             "n_candidate_edges", "n_fitted_edges_nonzero"]
meta_vals = Any["1.0", string(Dates.now(Dates.UTC)),
                "S_i(t) = sum_j alpha_ij * TF_j(t - tau)",
                DEFAULT_TAU, DEFAULT_W2, DEFAULT_MAX_NANS,
                MOLECULES_PER_CELL, "HiGHS via JuMP",
                "L1 on residuals + L1 penalty on alpha",
                "Teufel 2019 WT unstressed RPM",
                "Teufel 2019 supplementary table 4",
                n_sub, n_tf, n_candidate_edges, n_fitted_edges]

# README as a single column of text lines.
readme_lines = String.(split(README_STR, "\n"))

isfile(HANDOFF_XLSX) && rm(HANDOFF_XLSX)
XLSX.openxlsx(HANDOFF_XLSX, mode = "w") do xf
    sh = xf[1]
    XLSX.rename!(sh, "README")
    XLSX.writetable!(sh, [readme_lines], ["transcription_fit handoff"])

    sh = XLSX.addsheet!(xf, "metadata")
    XLSX.writetable!(sh, [meta_keys, meta_vals], ["key", "value"])

    sh = XLSX.addsheet!(xf, "alpha_matrix")
    XLSX.writetable!(sh, matrix_cols, matrix_labels)

    sh = XLSX.addsheet!(xf, "alpha_edges")
    XLSX.writetable!(sh, [edges_sub, edges_tf, edges_a],
                     ["substrate", "tf", "alpha"])

    sh = XLSX.addsheet!(xf, "tf_network_candidate")
    XLSX.writetable!(sh, [cand_sub, cand_tf], ["substrate", "tf"])

    sh = XLSX.addsheet!(xf, "tf_network_fitted")
    XLSX.writetable!(sh, [fit_sub, fit_tf, fit_a],
                     ["substrate", "tf", "alpha"])
end

println("Handoff written.")
println("  Folder:  ", HANDOFF_DIR)
println("  Zip:     ", HANDOFF_ZIP)
println("  Excel:   ", HANDOFF_XLSX)
println()
println("Counts:")
println("  ", n_sub, " substrates, ", n_tf, " TFs")
println("  ", n_candidate_edges, " candidate edges, ",
        n_fitted_edges, " fitted (nonzero alpha)")


## 14. What's not done yet (v2 +)

1. **Cross-validation per substrate.** Each LP has up to $2k + 36$
   variables and only 18 data points. For $k \geq 9$ we are
   over-parameterised in principle. Leave-one-timepoint-out CV would
   distinguish real fits from memorisation.
2. **Bootstrap confidence intervals on $\alpha$.** Resampling
   timepoints would give error bars and let us call out which
   $\alpha$ are robust.
3. **Random-network baseline.** Permuting the TF-substrate network and
   re-fitting tells us how much of the fit quality comes from real
   biology vs degrees of freedom. Important for the proposal / paper.
4. **Volume normalization.** mRNA molecules per cell scale with cell
   volume, which changes across the cycle. v1 ignores this; v2 should
   fold in a volume curve from the growth model.
5. **Joint substrate fitting.** Hierarchical formulation sharing
   residual scale or regularizer strength across substrates.
6. **Degradation term.** $dS_i / dt = \text{transcription} -
   a_{\text{deg}} S_i$. Strategy doc mentions this in the ODE form.
   Still an LP if $a_{\text{deg}}$ is per-substrate.
7. **Non-linear formulation.** Match the kinase / phospho-site
   structure from the Osaka slides: multiplicative regulator
   combinations. Loses convexity but may resolve the amplitude
   problem.
8. **Network completion.** The Teufel 2019 network has known holes
   (CLN2 ← MBP1 missing, for instance). Could augment with YEASTRACT
   or SGD-curated edges.

---

## Glossary

*Quick definitions for terms used throughout the notebook.*

**Biology and yeast cell cycle**

- **Cell cycle**: the four phases (G1, S, G2, M) a cell passes through to divide. In synchronized cultures, all cells are roughly at the same phase, which lets time-resolved RNA-seq trace gene expression dynamics.
- **Transcription factor (TF)**: a protein that binds DNA at gene promoters and modulates how often that gene gets transcribed. The set of TFs binding a given gene's promoter determines its transcription rate.
- **Substrate**: in this notebook, any gene being regulated by one or more TFs. Confusingly, in metabolic biology "substrate" means the molecule an enzyme acts on; here it just means "gene whose transcription rate we're modeling."
- **SBF complex**: Swi4 + Swi6, a TF dimer that drives G1/S transition genes (CLN1, CLN2, etc.).
- **MBF complex**: Mbp1 + Swi6, a related TF dimer that drives S-phase genes (CLB5, CLB6, etc.).
- **CLN1, CLN2, CLN3**: G1 cyclins. CLN3 acts early; CLN1/CLN2 act at G1/S.
- **CLB5, CLB6, CLB1, CLB2**: B-type cyclins. CLB5/6 in S phase, CLB1/2 in M.
- **alpha-factor**: a yeast mating pheromone. Treating MATa cells with alpha-factor arrests them in G1; removing it synchronizes the culture as cells exit. This is how Teufel got the synchronized timecourse.

**Data and units**

- **RPM (reads per million)**: RNA-seq normalization. Raw read counts per gene divided by total reads, times $10^6$. Removes library-size differences.
- **Molecules per cell**: estimated absolute copy number. We multiply RPM by $60{,}000 / 10^6 = 0.06$, where 60,000 is a rough estimate of total mRNA molecules per yeast cell.
- **Time axis**: minutes after alpha-factor washout (cells released from G1 arrest). Teufel measured at 0, 5, 10, ..., 150 min plus an "unsynchronized" baseline.

**Math and optimization**

- **LP (linear program)**: an optimization problem where both the objective and the constraints are linear functions of the decision variables. Solvable exactly and fast (polynomial time) for huge problem sizes.
- **L1 norm / L1 loss**: $\|x\|_1 = \sum |x_i|$. Sum of absolute values. Used as a loss makes the fit robust to outliers; used as a penalty drives small coefficients exactly to zero (sparsity).
- **L1 penalty**: $w \sum |\alpha_{ij}|$ added to the objective. The parameter $w$ controls how aggressively the fit zeros out small coefficients. We sweep over $w$ in section 9 to find a good default.
- **L-curve**: plot of fit quality (residual) vs solution complexity (max $|\alpha|$) across regularization strengths. The "elbow" of the curve marks the sweet spot where more regularization stops improving sparsity without costing much in fit.
- **Sparsity**: most coefficients being exactly zero. We want sparsity because biologically each gene is regulated by only a handful of TFs, not all of them.
- **L1 of the residual + L1 of $\alpha$**: makes the whole problem an LP. L2 (squared) loss would make it a quadratic program, also solvable but slower and not exact-sparse.
- **Vertex solution**: an LP's optimum sits at a vertex of the feasible polytope. With no regularization, the LP may pick a vertex with one giant coefficient instead of a balanced solution; the L1 penalty prevents that.

**Software**

- **JuMP**: Julia's standard modeling language for optimization. Lets you write LPs, MILPs, QPs etc. in math-like syntax.
- **HiGHS**: a fast open-source LP/MILP solver. JuMP delegates the actual solving to it.
- **Plots.jl**: Julia's most popular plotting package. We use it with the default GR backend.
- **DataFrames.jl + CSV.jl**: Julia's pandas-equivalent for tabular data.
- **XLSX.jl, ZipFile.jl**: used in section 13 to bundle the handoff folder.

**Project-specific**

- **Strategy doc**: `Strategy_TRX.docx`, the project specification that defines the linear model and L1+L1 objective. This notebook implements it directly.
- **The handoff**: the output folder `tf_handoff/` that section 13 writes. The artifact for the whole-cell rebuild team to consume.
- **Doc-literal formulation**: the model exactly as the strategy doc writes it: no intercept, no degradation term. We default to this; section 10 discusses the trade-offs.

## Appendix: re-deriving the CSV files

This deliverable ships both data files pre-converted to CSV so the
notebook is self-contained. If you ever need to re-derive them from
the originals:

### `WT_unstressed_readspermillionreads.csv` (from `.ods`)

```bash
pip install pandas odfpy
```

```python
import pandas as pd
df = pd.read_excel("WT_unstressed_readspermillionreads.ods", engine="odf")
df.columns = [str(c).strip() for c in df.columns]
df.to_csv("WT_unstressed_readspermillionreads.csv", index=False)
```

### `TF_von_Teufel.csv` (from `.numbers`)

**Option A (Apple Numbers app):** File → Export To → CSV. Save it as
`~/Downloads/TF_von_Teufel.csv` with `tf,relation,substrate` as the
column order (the export usually produces this anyway).

**Option B (Python):**

```bash
pip install numbers-parser
```

```python
from numbers_parser import Document
import csv
doc = Document("TF_von_Teufel.numbers")
tbl = doc.sheets[0].tables[0]
with open("TF_von_Teufel.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["tf", "relation", "substrate"])
    for r in range(tbl.num_rows):
        w.writerow([tbl.cell(r, c).value for c in range(tbl.num_cols)])
```

Either way, drop the resulting CSV into `~/Downloads/` and re-run.